# Task 2: Bayesian Change Point Modeling
## Brent Oil Change Point Analysis — Birhan Energies

This notebook builds a Bayesian change point model in PyMC to detect
structural breaks in Brent oil log prices, checks MCMC convergence,
interprets the posterior, and associates detected change points with the
researched key events.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

from scripts.data_loader import load_brent_prices, load_key_events
from scripts.event_matcher import match_change_point_to_events, summarize_impact

FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PyMC version:", pm.__version__)
print("ArviZ version:", az.__version__)


## 1. Load Data

We model `log_price` rather than raw `Price`. This keeps the scale of price
changes comparable across very different price regimes ($10s vs $100s), and
means a change in the model's mean parameter translates directly into a
*percentage* change in price — the natural unit for "impact" statements.

In [ ]:
prices = load_brent_prices()
events = load_key_events()

y = prices["log_price"].values
n = len(y)
dates = prices["Date"]

print(f"Series length: {n} observations")
print(f"Date range: {dates.min().date()} to {dates.max().date()}")


## 2. Build the Bayesian Change Point Model

Following the assignment spec:

- **Switch point `tau`**: discrete uniform prior over all possible day indices
- **Before/after means `mu_1`, `mu_2`**: one for each regime
- **Shared standard deviation `sigma`**: (a simplifying assumption for the
  core mandatory model — see the "Advanced Extensions" section below for a
  variance-shift version)
- **Switch function**: `pm.math.switch` selects `mu_1` or `mu_2` depending on
  whether the time index is before or after `tau`
- **Likelihood**: `pm.Normal`, connecting the switched mean to the observed
  `log_price` data

> **Note on runtime**: with ~9,000 observations, sampling this model can take
> several minutes on a laptop CPU. This is expected for PyMC's NUTS sampler
> on a series this long — let it run.

In [ ]:
with pm.Model() as changepoint_model:
    # Discrete uniform prior over the switch point (day index)
    tau = pm.DiscreteUniform("tau", lower=0, upper=n - 1)

    # Priors for the mean log-price before and after the switch point
    mu_1 = pm.Normal("mu_1", mu=y.mean(), sigma=y.std())
    mu_2 = pm.Normal("mu_2", mu=y.mean(), sigma=y.std())

    # Shared observation noise
    sigma = pm.HalfNormal("sigma", sigma=y.std())

    # Switch function: select mu_1 before tau, mu_2 from tau onward
    idx = np.arange(n)
    mu = pm.math.switch(idx >= tau, mu_2, mu_1)

    # Likelihood
    obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=y)

pm.model_to_graphviz(changepoint_model)


## 3. Run the Sampler (MCMC)

In [ ]:
with changepoint_model:
    trace = pm.sample(
        draws=2000,
        tune=1500,
        chains=4,
        cores=4,
        target_accept=0.9,
        random_seed=42,
        return_inferencedata=True,
    )


## 4. Check Convergence

- `r_hat` should be close to 1.0 (values above ~1.01 suggest the chains have
  not converged/mixed well)
- Trace plots should look like "fuzzy caterpillars" with no trend and
  overlapping chains

In [ ]:
summary = az.summary(trace, var_names=["tau", "mu_1", "mu_2", "sigma"])
summary


In [ ]:
az.plot_trace(trace, var_names=["tau", "mu_1", "mu_2", "sigma"])
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_trace_plot.png", dpi=130)
plt.show()


## 5. Identify the Change Point

The posterior distribution of `tau` tells us *when* the model believes the
structural break occurred, and how confident it is (a sharp, narrow peak =
high confidence; a wide/flat posterior = low confidence, possibly multiple
plausible break points).

In [ ]:
tau_samples = trace.posterior["tau"].values.flatten()
tau_mode = int(pd.Series(tau_samples).mode()[0])
change_point_date = dates.iloc[tau_mode]

print(f"Posterior mode of tau: index {tau_mode}")
print(f"Corresponding date: {change_point_date.date()}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(tau_samples, bins=100, color="#1f4e79")
ax.axvline(tau_mode, color="crimson", linestyle="--", label=f"Mode: {change_point_date.date()}")
ax.set_title("Posterior Distribution of the Change Point (tau)")
ax.set_xlabel("Day index")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_tau_posterior.png", dpi=130)
plt.show()


## 6. Quantify the Impact

Posterior distributions for `mu_1` and `mu_2` let us make probabilistic
statements about the size of the shift, not just a single point estimate.

In [ ]:
mu_1_samples = trace.posterior["mu_1"].values.flatten()
mu_2_samples = trace.posterior["mu_2"].values.flatten()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(np.exp(mu_1_samples), bins=60, alpha=0.6, label="Before (mu_1)", color="#1f4e79")
ax.hist(np.exp(mu_2_samples), bins=60, alpha=0.6, label="After (mu_2)", color="#8b0000")
ax.set_title("Posterior Price Distributions Before vs. After the Change Point")
ax.set_xlabel("Price (USD/barrel)")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_before_after_price.png", dpi=130)
plt.show()

# Probabilistic statement
prob_increase = (mu_2_samples > mu_1_samples).mean()
print(f"Posterior probability that the mean price increased after the change point: {prob_increase:.1%}")
print()
print(summarize_impact(mu_1_samples.mean(), mu_2_samples.mean(), is_log=True))


## 7. Associate the Change Point with Researched Events

Compare the detected change point date against `data/raw/key_events.csv` to
find plausible real-world triggers within a tolerance window.

In [ ]:
matches = match_change_point_to_events(change_point_date, events, window_days=45)

if len(matches) > 0:
    print(f"Candidate event(s) near the detected change point ({change_point_date.date()}):\n")
    print(matches[["event_name", "start_date", "category", "days_from_change_point"]])
else:
    print(f"No researched event found within the tolerance window of {change_point_date.date()}.")
    print("Consider widening window_days, or note this as an unexplained/undocumented shift.")


## 8. Interpretation Summary

Fill in based on the actual run output above. Example template:

> The model detects a change point around **[DATE]** with r_hat values at
> 1.00 for all parameters, indicating good convergence. The posterior for
> `tau` shows a [sharp / moderately wide] peak, indicating [high / moderate]
> confidence in the timing. The average price shifted from **$X** to **$Y**
> (a Z% [increase/decrease]), with [N]% posterior probability of an increase.
> This aligns with **[EVENT NAME]** on [EVENT DATE], [N] days from the
> detected change point.
>
> **Caveat**: this is a statistical association, not proof of causation — see
> `reports/task1_workflow_and_assumptions.md` section 5 for the full
> discussion of this limitation.

## 9. Advanced Extensions (Optional — for Future Work section)

The mandatory model above detects a **single** change point in the **mean**
of log price, assuming constant variance. Real Brent oil price dynamics are
richer than this. Possible extensions, not required for the core submission:

1. **Multiple change points**: extend `tau` to a vector of switch points (or
   use a marked Poisson process prior) to segment the full 35-year series
   into several regimes instead of just two.
2. **Variance change point**: add a second switch function on `sigma` (or
   model `sigma_1`, `sigma_2` separately) to detect shifts in volatility, not
   just the mean — motivated by the volatility clustering seen in Task 1's
   EDA (2008, 2014-16, 2020 all show visibly different volatility regimes).
3. **VAR (Vector Autoregression)**: incorporate macro variables (GDP growth,
   inflation, USD exchange rate index) to study dynamic relationships between
   oil prices and the broader economy, rather than univariate price alone.
4. **Markov-Switching models**: explicitly define latent 'calm' vs 'volatile'
   market regimes with estimated transition probabilities, allowing the
   series to switch back and forth between regimes rather than assuming a
   one-way structural break.
5. **Additional data sources**: OPEC spare capacity, US shale rig counts, and
   strategic petroleum reserve levels could sharpen the supply-side story
   alongside the demand/macro variables above.